# Cross Validation

The goal of the cross-validation done in this notebook is to evaluate the model's performance using only the training data, to avoid overfitting to the test set. This provides a more robust evaluation of the model's performance without using data that has already been reserved for the final test.

After performing cross-validation and selecting the models with the best performance, we can test it on the test set, which is used only for the final evaluation.

Therefore, cross-validation uses only the training data, which helps optimize the model without "leaking" test data.

Before testing the chosen models, we also evaluate how removing the weakly corrolated features impacts these models. If the impact is positive, the models will be tested without those features.

In [14]:
# Import libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, ClassifierMixin
import xgboost as xgb 

# Define the file path relative to the current notebook directory
base_path = os.path.join("..", "data", "processed")

# Verificar a existência dos arquivos de dados
files = ["X_train_scaled.csv", "y_efficiency_train.csv", "y_efficiency_level_train.csv"]
for file in files:
    if not os.path.exists(os.path.join(base_path, file)):
        raise FileNotFoundError(f"The file {file} does not exist in the directory {base_path}. Please check the path and try again.")

# Load preprocessed data
X_train = pd.read_csv(os.path.join(base_path, "X_train_scaled.csv"))
y_efficiency_train = pd.read_csv(os.path.join(base_path, "y_efficiency_train.csv")).values.ravel()
y_efficiency_level_train = pd.read_csv(os.path.join(base_path, "y_efficiency_level_train.csv")).values.ravel()
y_efficiency_level_train_int = pd.read_csv(os.path.join(base_path, "y_efficiency_level_train_int.csv")).values.ravel()

# Define K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# kNN Regressor
knn_regressor = KNeighborsRegressor(n_neighbors=5)
knn_regressor_scores = cross_val_score(knn_regressor, X_train, y_efficiency_train, cv=kf, scoring='neg_mean_squared_error')

# kNN Classifier
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier_scores = cross_val_score(knn_classifier, X_train, y_efficiency_level_train, cv=kf, scoring='accuracy')

# Linear Regression
linear_regressor = LinearRegression()
linear_regressor_scores = cross_val_score(linear_regressor, X_train, y_efficiency_train, cv=kf, scoring='neg_mean_squared_error')

# Naive Bayes
# Custom hybrid Naive Bayes model
class HybridNaiveBayes(BaseEstimator, ClassifierMixin):
    def __init__(self):
        self.gnb = GaussianNB()
        self.bnb = BernoulliNB()

    def fit(self, X, y):
        # Separate continuous and binary features
        self.X_train_continuous = X.iloc[:, :5]  # Adjust columns based on actual data
        self.X_train_binary = X.iloc[:, 5:]  # Adjust columns based on actual data
        self.y_train = y

        # Fit the models
        self.gnb.fit(self.X_train_continuous, self.y_train)
        self.bnb.fit(self.X_train_binary, self.y_train)

        # Set the class labels (required for scoring in cross-validation)
        self.classes_ = self.gnb.classes_

        return self

    def predict(self, X):
        # Separate continuous and binary features
        X_test_continuous = X.iloc[:, :5]  # Adjust columns based on actual data
        X_test_binary = X.iloc[:, 5:]  # Adjust columns based on actual data

        # Predict probabilities for each model
        proba_continuous = self.gnb.predict_proba(X_test_continuous)
        proba_binary = self.bnb.predict_proba(X_test_binary)

        # Ensure class labels are consistent between the models
        assert np.array_equal(self.gnb.classes_, self.bnb.classes_), "Class labels must match!"

        # Combine probabilities using Naive Bayes Theorem (multiplication)
        class_priors = self.gnb.class_prior_  # Use priors from GaussianNB
        mult_proba = proba_continuous * proba_binary
        mult_proba *= class_priors  # Multiply by prior probabilities
        mult_proba /= mult_proba.sum(axis=1, keepdims=True)  # Normalize to ensure probabilities sum to 1

        # Make predictions based on the combined probabilities
        y_pred = np.argmax(mult_proba, axis=1)
        return self.gnb.classes_[y_pred]

    def predict_proba(self, X):
        # Separate continuous and binary features
        X_test_continuous = X.iloc[:, :5]  # Adjust columns based on actual data
        X_test_binary = X.iloc[:, 5:]  # Adjust columns based on actual data

        # Predict probabilities for each model
        proba_continuous = self.gnb.predict_proba(X_test_continuous)
        proba_binary = self.bnb.predict_proba(X_test_binary)

        # Combine probabilities using Naive Bayes Theorem (multiplication)
        class_priors = self.gnb.class_prior_  # Use priors from GaussianNB
        mult_proba = proba_continuous * proba_binary
        mult_proba *= class_priors  # Multiply by prior probabilities
        mult_proba /= mult_proba.sum(axis=1, keepdims=True)  # Normalize to ensure probabilities sum to 1

        return mult_proba
    
hybrid_nb = HybridNaiveBayes()
hybrid_nb_scores = cross_val_score(hybrid_nb, X_train, y_efficiency_level_train, cv=kf, scoring='accuracy')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_scores = cross_val_score(rf, X_train, y_efficiency_level_train, cv=kf, scoring='accuracy')

# Logistic Regression (for classification)
logistic_regressor = LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000)
logistic_regressor_scores = cross_val_score(logistic_regressor, X_train, y_efficiency_level_train, cv=kf, scoring='accuracy')

# XGBoost Regressor (for regression)
xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
xgb_regressor_scores = cross_val_score(xgb_regressor, X_train, y_efficiency_train, cv=kf, scoring='neg_mean_squared_error')

# XGBoost Classifier (for classification)
xgb_classifier = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False  # Avoids unnecessary warnings
)
xgb_classifier_scores = cross_val_score(xgb_classifier, X_train, y_efficiency_level_train_int, cv=kf, scoring='accuracy')

# Print results
print("kNN Regressor CV Scores:", knn_regressor_scores)
print("kNN Classifier CV Scores:", knn_classifier_scores)
print("Linear Regressor CV Scores:", linear_regressor_scores)
print("HybridNB CV Scores:", hybrid_nb_scores)
print("Random Forest CV Scores:", rf_scores)
print("Logistic Regression CV Scores:", logistic_regressor_scores)
print("XGBoost Regressor CV Scores:", xgb_regressor_scores)
print("XGBoost Classifier CV Scores:", xgb_classifier_scores)

c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:31:53] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:31:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:32:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda

kNN Regressor CV Scores: [-1.93193981e-05 -1.95035089e-05 -1.94554412e-05 -1.96300682e-05
 -1.95273698e-05]
kNN Classifier CV Scores: [0.8685625 0.8751875 0.8755    0.8736875 0.8715625]
Linear Regressor CV Scores: [-1.39234513e-06 -1.47424951e-06 -1.42911801e-06 -1.40523562e-06
 -1.47917206e-06]
HybridNB CV Scores: [0.7423125 0.7463125 0.752     0.7438125 0.748375 ]
Random Forest CV Scores: [0.9810625 0.9809375 0.982     0.98      0.97975  ]
Logistic Regression CV Scores: [0.9873125 0.9883125 0.986875  0.9880625 0.988375 ]
XGBoost Regressor CV Scores: [-3.54840324e-07 -4.53045673e-07 -4.15523072e-07 -3.97667096e-07
 -3.89609067e-07]
XGBoost Classifier CV Scores: [0.9884375 0.9903125 0.9905    0.989625  0.98875  ]


## Analysis of Cross-Validation Results
Based on the cross-validation results, here is what we concluded about the models' performance:

- **kNN Regressor:**
    The negative Mean Squared Error (MSE) values indicate that the kNN regression model is performing well, with very low error, which is consistent across the cross-validation folds.

- **kNN Classifier:**
    The accuracy of the kNN classifier is consistent and high, around 86%, which suggests that the kNN classifier is performing well in the classification task.

- **Linear Regression and XGBoost Regression:**
    The very low Mean Squared Error (MSE) values indicate that both the linear and XGBoost regression models are performing extremely well, with negligible error.
    The R² value being very close to 1 suggests that the models explain almost all the variability of the response data around its mean.

- **Naive Bayes:**
    The Hybrid Naive Bayes has an accuracy of approximately 73-75%, which is reasonable but not as high as the kNN classifier or Random Forest.

- **Random Forest:**
    The Random Forest model shows very high accuracy, around 98%, suggesting it is a very effective model for this classification task.
    The consistency of the results across the cross-validation folds indicates that the model is robust.

- **Logistic Regression (softmax):**
    The softmax model shows very high accuracy, around 98.8%, suggesting it is a very effective model for this classification task, possibly being better than the random forest.
    The consistency of the results across the cross-validation folds indicates that the model is robust.

- **XGBoost Classifier:**
    The XGBoost classifier model had the highest accuracies, around 99%, suggesting it might be the best model for this classification task
    The consistency of the results across the cross-validation folds indicates that the model is robust.


### Conclusion
- Based on the cross-validation results, we can conclude that the **Random Forest model**, **Logistic Regression model** and **Xgboost classifier model** are the most effective for the **classification task**, therefore we will train these models on a training set and evaluate their performance on the test set. 

- For the **regression task**, both the Linear Regression and XGBoost Regressor are performing very well, so both of them will be trained and evaluated to compare their performace.

## Removing Weakly Correlated Features

As stated in the EDA notebook, we'll try to remove the weakly correlated features and analyse how the impact the relevant models performances

In [15]:
# Remove weakly correlated features
features_to_keep = ['Isc', 'irradiance', 'affected_area', 'Voc', 'pv_module_type_Thin-Film', 'pv_module_type_HJT', 'temperature']
X_train_updated = X_train.loc[:, features_to_keep]

# Linear Regression
linear_regressor = LinearRegression()
new_linear_regressor_scores = cross_val_score(linear_regressor, X_train_updated, y_efficiency_train, cv=kf, scoring='neg_mean_squared_error')

# Logistic Regression (for classification)
logistic_regressor = LogisticRegression(random_state=42, solver='lbfgs', max_iter=1000)
new_logistic_regressor_scores = cross_val_score(logistic_regressor, X_train_updated, y_efficiency_level_train, cv=kf, scoring='accuracy')

# XGBoost Regressor (for regression)
xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
new_xgb_regressor_scores = cross_val_score(xgb_regressor, X_train_updated, y_efficiency_train, cv=kf, scoring='neg_mean_squared_error')

# XGBoost Classifier (for classification)
xgb_classifier = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False  # Avoids unnecessary warnings
)
new_xgb_classifier_scores = cross_val_score(xgb_classifier, X_train_updated, y_efficiency_level_train_int, cv=kf, scoring='accuracy')

# Print results
print("Linear Regressor CV Scores:", linear_regressor_scores)
print("Logistic Regression CV Scores:", logistic_regressor_scores)
print("XGBoost Regressor CV Scores:", xgb_regressor_scores)
print("XGBoost Classifier CV Scores:", xgb_classifier_scores)

print("New Linear Regressor CV Scores:", new_linear_regressor_scores)
print("New Logistic Regression CV Scores:", new_logistic_regressor_scores)
print("New XGBoost Regressor CV Scores:", new_xgb_regressor_scores)
print("New XGBoost Classifier CV Scores:", new_xgb_classifier_scores)

c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:32:58] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:33:03] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [18:33:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0c55ff5f71b100e98-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\Utilizador\anaconda

Linear Regressor CV Scores: [-1.39234513e-06 -1.47424951e-06 -1.42911801e-06 -1.40523562e-06
 -1.47917206e-06]
Logistic Regression CV Scores: [0.9873125 0.9883125 0.986875  0.9880625 0.988375 ]
XGBoost Regressor CV Scores: [-3.54840324e-07 -4.53045673e-07 -4.15523072e-07 -3.97667096e-07
 -3.89609067e-07]
XGBoost Classifier CV Scores: [0.9884375 0.9903125 0.9905    0.989625  0.98875  ]
New Linear Regressor CV Scores: [-1.39362118e-06 -1.47643670e-06 -1.43211661e-06 -1.40629585e-06
 -1.48119434e-06]
New Logistic Regression CV Scores: [0.9446875 0.94275   0.9479375 0.94575   0.9449375]
New XGBoost Regressor CV Scores: [-4.16366364e-07 -4.95541224e-07 -4.70242649e-07 -4.48040374e-07
 -4.37877763e-07]
New XGBoost Classifier CV Scores: [0.987125  0.987125  0.989     0.9878125 0.987375 ]


## Removing features analysis

The analysis of removing features revealed a slight decrease in accuracies across the models, indicating that feature removal negatively impacts performance. Consequently, it may not be advisable to eliminate these features unless reducing training time of the models if very important.